# 02 投資配分の比較

均等配分と 4 つの重点配分（教育・研究開発・インフラ・医療）を、同じ予算・同じ期間で比べます（RQ1・RQ2）。
**どの配分が最もよいかは決めていません**。比べる指標によって答えが変わるかを調べます。

> **補遺トピック G1・教育用の架空モデル**です。国名・経済指標・投資効果の係数はすべて架空で、現実の国家の GDP・経済成長率・政策効果を予測するものではありません。実際の政策判断に使わないでください。

## 準備：実験環境（最初に必ず実行）

G1 リポジトリの `.py`（計算の本体）を import できるように、作業ディレクトリ（G1 のルート）を自動で探して設定します。

| 実行場所 | 準備 |
| --- | --- |
| ローカルの Jupyter / VSCode | `notebooks/` の中で開けば、そのまま動きます |
| Colab（方法A：GitHub から clone） | 下のセルの `REPO_URL` に**自分たちのリポジトリの URL** を入れる |
| Colab（方法B：ZIP をアップロード） | G1 フォルダを ZIP にして `/content` にアップロードしておく（またはセルの実行中にアップロード） |

Colab の CPU ランタイムで動きます（GPU・TPU は不要）。`textbook/`（教科書のコード）は公開リポジトリに含めない約束です。`textbook/` がないと import が失敗します。

In [ ]:
#@title 準備（G1 の .py を読み込む）
import os, sys, glob, zipfile, subprocess
from pathlib import Path

# ---- 方法A：GitHub から clone する場合だけ設定する（URL は自分たちのものを入れる）----
REPO_URL = ""                                     # 例："https://github.com/<team>/<repo>.git"（空なら clone しない）
REPO_SUBDIR = "g1-economic-growth-optimization"   # リポジトリの中の G1 フォルダ（ルートが G1 なら ""）

IN_COLAB = "google.colab" in sys.modules


def is_g1_root(p):
    return (Path(p) / "economic_model.py").exists() and (Path(p) / "optimization.py").exists()


def find_g1_root():
    for p in [Path.cwd(), Path.cwd().parent]:          # ローカル：notebooks/ の中、または G1 のルート
        if is_g1_root(p):
            return p.resolve()
    if not IN_COLAB:
        return None
    if REPO_URL:                                        # Colab 方法A
        dest = Path("/content/repo")
        if not dest.exists():
            subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(dest)], check=True)
        cand = dest / REPO_SUBDIR if REPO_SUBDIR else dest
        if is_g1_root(cand):
            return cand
    zips = glob.glob("/content/*.zip")                  # Colab 方法B
    if not zips:
        from google.colab import files
        print("G1 フォルダの ZIP を選んでアップロードしてください")
        zips = ["/content/" + name for name in files.upload()]
    for z in zips:
        with zipfile.ZipFile(z) as zf:
            zf.extractall("/content/g1_upload")
    for hit in glob.glob("/content/g1_upload/**/economic_model.py", recursive=True):
        if is_g1_root(Path(hit).parent):
            return Path(hit).parent
    return None


G1_ROOT = find_g1_root()
if G1_ROOT is None:
    raise RuntimeError("G1 のフォルダが見つかりません。上の説明（方法A／方法B）を確認してください。")
os.chdir(G1_ROOT)
if str(G1_ROOT) not in sys.path:
    sys.path.insert(0, str(G1_ROOT))
if not (G1_ROOT / "textbook" / "fixed_queue.py").exists():
    print("⚠️ textbook/ がありません。economic_model.py・optimization.py の import が失敗します。")

# ---- 既存の .py（計算の本体）と、グラフ ----
import matplotlib.pyplot as plt
import economic_model as em
import simulation as sim
import allocation_strategies as al
import optimization as op
import evaluation as ev
import growth_charts as gc

gc.setup_style()                        # 文字を大きくするなら gc.setup_style(1.3)
PARAMS = em.load_economy()              # 架空の国家のパラメータ
SCENARIOS = al.load_scenarios()         # 比較用の配分シナリオ
LABELS = {sc["id"]: sc["label"] for sc in SCENARIOS}
YEARS = PARAMS["simulation_years"]
STEP = 0.1                              # 最適化の配分の刻み（10%）
M = al.units_count(STEP)

SAVE_FIGURES = False                    # True にすると、グラフを notebooks/figures/ に PNG で保存する
FIG_DIR = "notebooks/figures"


def show(fig, name):
    """グラフを表示し、SAVE_FIGURES = True なら PNG も保存する"""
    if SAVE_FIGURES:
        print("saved:", gc.save(fig, f"{FIG_DIR}/{name}.png"))
    plt.show()


def table(rows, columns):
    """辞書のリストを表で表示する。columns = [(キー, 見出し, 書式)]。pandas があれば DataFrame で表示"""
    try:
        import pandas as pd
        from IPython.display import display
        display(pd.DataFrame([{h: ("-" if r.get(k) is None else f.format(r[k])) for k, h, f in columns} for r in rows]))
    except ImportError:
        print(ev.format_table(rows, columns))


def alloc_cols():
    return [(s, em.SECTOR_LABELS[s], "{:.0%}") for s in em.SECTORS]


print("G1_ROOT:", G1_ROOT)
print(PARAMS["country"], "｜", YEARS, "年、年間予算", PARAMS["annual_investment_budget"], "（架空の単位）")
print("シナリオ:", [sc["label"] for sc in SCENARIOS])

## 1. 投資配分（予算制約：配分率の合計 = 1）

In [ ]:
#@title 投資配分の積み上げ棒グラフ
show(gc.plot_allocation_bars([(LABELS[sc["id"]].replace(" ", "\n", 1), sc["shares"]) for sc in SCENARIOS],
                             "Investment allocation scenarios"), "02_allocation")

## 2. GDP 成長曲線

In [ ]:
#@title GDP 成長曲線
runs = sim.run_scenarios(SCENARIOS, PARAMS)
show(gc.plot_series(runs, "gdp", "GDP by allocation scenario (model)", "GDP (fictional units)", labels=LABELS),
     "02_gdp")

## 3. GDP と生活水準の比較

In [ ]:
#@title GDP と生活水準の比較
rows = [{"name": LABELS[k], **ev.summarize_run(r)} for k, r in runs.items()]
table(rows, [("name", "scenario", "{}"), ("final_gdp", "final GDP", "{:.1f}"), ("gdp_growth_annual", "GDP / year", "{:.2%}"),
             ("cumulative_gdp", "cumulative GDP", "{:.0f}"), ("final_gdp_per_capita", "GDP per capita", "{:.2f}"),
             ("final_living_standard", "living standard", "{:.3f}")])
names = [LABELS[k].replace(" ", "\n", 1) for k in runs]
show(gc.plot_grouped_bars(names, ["GDP index", "living standard"],
                          {"GDP index": [r["final_gdp"] / PARAMS["initial_gdp"] for r in rows],
                           "living standard": [r["final_living_standard"] for r in rows]},
                          "Final GDP index vs living standard (both: year 0 = 1, model)", "index (year 0 = 1)",
                          colors={"GDP index": "#3d3c39", "living standard": "#6250d6"}), "02_gdp_vs_ls")

## 4. 分野別の状態変数（最終年）

In [ ]:
#@title 分野別の状態変数
states = ["human_capital", "technology", "infrastructure", "health"]
show(gc.plot_grouped_bars([gc.STATE_LABELS[s] for s in states], [LABELS[k] for k in runs],
                          {LABELS[k]: [r[-1][s] for s in states] for k, r in runs.items()},
                          f"State variables in year {YEARS} (initial = 1, model)", "index (initial = 1)",
                          colors={LABELS[k]: gc.SCENARIO_COLORS[k] for k in runs}), "02_states")

## 5. 学生向け TODO：新しい配分シナリオ

In [ ]:
# TODO 1：配分率を変更する / TODO 2：新しい配分シナリオを作る（合計 1 にする）
my = {"education": 0.35, "rd": 0.15, "infrastructure": 0.15, "healthcare": 0.35}
sim.validate_allocation(my)                       # 合計が 1 でなければエラー
mine = sim.simulate(my, PARAMS)
# TODO 3：GDP と生活水準を比較する
for name, r in [("equal", runs["equal"]), ("my scenario", mine)]:
    s = ev.summarize_run(r)
    print(f"{name:12}: GDP {s['final_gdp']:8.1f}  生活水準 {s['final_living_standard']:.3f}  健康 {s['final_health']:.2f}")

## 考察（チームで記入）

- **TODO**：GDP が最も大きいシナリオと、生活水準が最も高いシナリオは同じか（RQ2）
- **TODO**：重点配分は、他の分野の状態変数にどう影響したか（予算制約の下でのトレードオフ）
- **TODO**：「均等配分」がよく見えるとしたら、それはモデルのどの仮定（収穫逓減など）のためか

## 参考文献

- 柴田望洋『新・明解 Pythonで学ぶアルゴリズムとデータ構造（第2版）』SBクリエイティブ（第2章 最大値、第3章 線形探索、第4章 キュー、第6章 単純挿入ソート）
- G1 リポジトリの `README.md`（モデルの仮定・計算量・モデルの限界）
- データ：`data/baseline_economy.json`、`data/investment_scenarios.json`（この教材のための**架空のデータ**）